In [1]:
import configparser
import os

import pandas as pd

In [ ]:
os.chdir('../')

In [2]:
dataset = 'all_noHypoHyper'

In [3]:
data = configparser.ConfigParser(interpolation=configparser.ExtendedInterpolation())
data.read('../data.ini')
data = data['DEFAULT']

## Load

In [4]:
dataMutations = pd.read_csv('../' + data[f'{dataset}___mutations'], sep='\t', comment='#', engine='python')
dataMutations_RAS = pd.read_csv('../' + data[f'{dataset}_RAS___mutations'], sep='\t', comment='#', engine='python')
dataMutations_nonRAS = pd.read_csv('../' + data[f'{dataset}_nonRAS___mutations'], sep='\t', comment='#', engine='python')

In [5]:
dataClinicalSample = pd.read_csv('../' + data[f'{dataset}___samples'], sep='\t', comment='#', engine='python')
dataClinicalSample_RAS = pd.read_csv('../' + data[f'{dataset}_RAS___samples'], sep='\t', comment='#', engine='python')
dataClinicalSample_nonRAS = pd.read_csv('../' + data[f'{dataset}_nonRAS___samples'], sep='\t', comment='#', engine='python')

In [6]:
mutsig = pd.read_csv("../" + data[f'mutsig___{dataset}___out_dir'] + '/sig_genes.txt', sep='\t', engine='python', comment='#')
mutsig_RAS = pd.read_csv("../" + data[f'mutsig___{dataset}_RAS___out_dir'] + '/sig_genes.txt', sep='\t', engine='python', comment='#')
mutsig_nonRAS = pd.read_csv("../" + data[f'mutsig___{dataset}_nonRAS___out_dir'] + '/sig_genes.txt', sep='\t', engine='python', comment='#')

In [7]:
synonyms = pd.read_csv("../" + data['mutsig'] + "/synonyms_final.csv", sep='\t')
synonyms

,mutsig_name,genes,genes ∈ ALL,genes ∈ (ALL \ MUTSIG),#(genes),#(genes ∈ ALL),#(genes ∈ (ALL \ MUTSIG)),MANUALLY_RENAMED,choosen
0,A2BP1,RBFOX1,RBFOX1,RBFOX1,1,1,1,NaN,RBFOX1
1,A2LD1,GGACT,GGACT,GGACT,1,1,1,NaN,GGACT
2,AARSD1,AARSD1,NaN,NaN,1,0,0,NaN,NaN
3,ABO,ABO,NaN,NaN,1,0,0,NaN,NaN
4,ABP1,AOC1 | DBNL,AOC1 | DBNL,AOC1,2,2,1,NaN,AOC1
...,...,...,...,...,...,...,...,...,...
1468,ZNF826,ZNF826P,NaN,NaN,1,0,0,NaN,NaN
1469,ZNF828,CHAMP1,CHAMP1,CHAMP1,1,1,1,NaN,CHAMP1
1470,ZNF84,ZNF84,NaN,NaN,1,0,0,NaN,NaN
1471,ZNF883,ZNF883,NaN,NaN,1,0,0,NaN,NaN


In [8]:
rename_map = synonyms[synonyms['choosen'].notna()][['mutsig_name', 'choosen']].set_index('mutsig_name').to_dict()['choosen']

In [9]:
mutsig['gene'] = mutsig['gene'].replace(rename_map)
mutsig_RAS['gene'] = mutsig_RAS['gene'].replace(rename_map)
mutsig_nonRAS['gene'] = mutsig_nonRAS['gene'].replace(rename_map)

## Frequencies

In [10]:
# TODO: count more times the same sample or not ??
patientsGenes = pd.merge(dataMutations, dataClinicalSample, left_on='Tumor_Sample_Barcode', right_on='SAMPLE_ID')[['PATIENT_ID', 'Hugo_Symbol']].drop_duplicates().reset_index(drop=True)
patientsGenes_RAS = pd.merge(dataMutations_RAS, dataClinicalSample_RAS, left_on='Tumor_Sample_Barcode', right_on='SAMPLE_ID')[['PATIENT_ID', 'Hugo_Symbol']].drop_duplicates().reset_index(drop=True)
patientsGenes_nonRAS = pd.merge(dataMutations_nonRAS, dataClinicalSample_nonRAS, left_on='Tumor_Sample_Barcode', right_on='SAMPLE_ID')[['PATIENT_ID', 'Hugo_Symbol']].drop_duplicates().reset_index(drop=True)

In [11]:
numPatients_perGene = patientsGenes.groupby('Hugo_Symbol').count().reset_index().rename(columns={'PATIENT_ID': 'Number_Patients'})
numPatients_perGene_RAS = patientsGenes_RAS.groupby('Hugo_Symbol').count().reset_index().rename(columns={'PATIENT_ID': 'Number_Patients'})
numPatients_perGene_nonRAS = patientsGenes_nonRAS.groupby('Hugo_Symbol').count().reset_index().rename(columns={'PATIENT_ID': 'Number_Patients'})

In [12]:
numPatients = dataClinicalSample['PATIENT_ID'].nunique()
numPatients

1119

In [13]:
numPatients_perGene['Mutation_Frequency'] = numPatients_perGene['Number_Patients'] / numPatients
numPatients_perGene = numPatients_perGene.sort_values(by='Mutation_Frequency', ascending=False).reset_index(drop=True)

numPatients_perGene_RAS['Mutation_Frequency'] = numPatients_perGene_RAS['Number_Patients'] / numPatients
numPatients_perGene_RAS = numPatients_perGene_RAS.sort_values(by='Mutation_Frequency', ascending=False).reset_index(drop=True)

numPatients_perGene_nonRAS['Mutation_Frequency'] = numPatients_perGene_nonRAS['Number_Patients'] / numPatients
numPatients_perGene_nonRAS = numPatients_perGene_nonRAS.sort_values(by='Mutation_Frequency', ascending=False).reset_index(drop=True)

In [14]:
numPatients_perGene

,Hugo_Symbol,Number_Patients,Mutation_Frequency
0,APC,740,0.661305
1,TP53,631,0.563896
2,TTN,605,0.540661
3,KRAS,407,0.363718
4,SYNE1,340,0.303843
...,...,...,...
18803,TMEM158,1,0.000894
18804,KRTAP3-3,1,0.000894
18805,ATOH7,1,0.000894
18806,CBWD5,1,0.000894


In [15]:
frequencies = numPatients_perGene[['Hugo_Symbol', 'Mutation_Frequency']]
frequencies_RAS = numPatients_perGene_RAS[['Hugo_Symbol', 'Mutation_Frequency']]
frequencies_nonRAS = numPatients_perGene_nonRAS[['Hugo_Symbol', 'Mutation_Frequency']]

In [16]:
frequencies.to_csv("../" + data[f'hotnet2___{dataset}___in_dir'] + '/mutation_frequencies.csv', sep=' ', index=False, header=False)
frequencies_RAS.to_csv("../" + data[f'hotnet2___{dataset}_RAS___in_dir'] + '/mutation_frequencies.csv', sep=' ', index=False, header=False)
frequencies_nonRAS.to_csv("../" + data[f'hotnet2___{dataset}_nonRAS___in_dir'] + '/mutation_frequencies.csv', sep=' ', index=False, header=False)

# Mutsig scores

In [17]:
mutsig_p = mutsig[['gene', 'q']]
mutsig_p_RAS = mutsig_RAS[['gene', 'q']]
mutsig_p_nonRAS = mutsig_nonRAS[['gene', 'q']]

In [18]:
mutsig_q = mutsig[['gene', 'q']]
mutsig_q_RAS = mutsig_RAS[['gene', 'q']]
mutsig_q_nonRAS = mutsig_nonRAS[['gene', 'q']]

In [19]:
mutsig_p.to_csv("../" + data[f'hotnet2___{dataset}___in_dir'] + '/mutsig_pvalues.csv', sep=' ', index=False, header=False, float_format='%.15f')
mutsig_p_RAS.to_csv("../" + data[f'hotnet2___{dataset}_RAS___in_dir'] + '/mutsig_pvalues.csv', sep=' ', index=False, header=False, float_format='%.15f')
mutsig_p_nonRAS.to_csv("../" + data[f'hotnet2___{dataset}_nonRAS___in_dir'] + '/mutsig_pvalues.csv', sep=' ', index=False, header=False, float_format='%.15f')

In [20]:
mutsig_q.to_csv("../" + data[f'hotnet2___{dataset}___in_dir'] + '/mutsig_qvalues.csv', sep=' ', index=False, header=False, float_format='%.15f')
mutsig_q_RAS.to_csv("../" + data[f'hotnet2___{dataset}_RAS___in_dir'] + '/mutsig_qvalues.csv', sep=' ', index=False, header=False, float_format='%.15f')
mutsig_q_nonRAS.to_csv("../" + data[f'hotnet2___{dataset}_nonRAS___in_dir'] + '/mutsig_qvalues.csv', sep=' ', index=False, header=False, float_format='%.15f')